# Unsupervised Learning: Clustering, Density Estimation & Manifold Learning

Unsupervised learning discovers latent structure, clusters, low-dimensional manifolds, and anomalous patterns in unlabeled datasets $\mathcal{D} = \{\mathbf{x}_i\}_{i=1}^N$ without supervising ground truth signals.

### Table of Contents
1. [K-Means vs. DBSCAN: Geometric Clustering Limits](#1-k-means-vs-dbscan-geometric-clustering-limits)
2. [Gaussian Mixture Models (GMM) & Soft Clustering](#2-gaussian-mixture-models-gmm--soft-clustering)
3. [Linear vs. Non-Linear Dimensionality Reduction: PCA vs. t-SNE](#3-linear-vs-non-linear-dimensionality-reduction-pca-vs-t-sne)
4. [Unsupervised Anomaly Detection with Isolation Forests](#4-unsupervised-anomaly-detection-with-isolation-forests)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import IsolationForest

sns.set_theme(style="whitegrid")
print("Loaded Unsupervised Learning interactive environment.")

## 1. K-Means vs. DBSCAN: Geometric Clustering Limits
- **K-Means**: Assumes spherical, isotropic clusters with equal variance. Minimizes within-cluster sum of squares.
- **DBSCAN**: Density-based clustering. Connects core points with dense $\epsilon$-neighborhoods, discovering arbitrary non-convex geometries and isolating noise.

In [ ]:
X_moons, _ = make_moons(n_samples=300, noise=0.06, random_state=42)

km = KMeans(n_clusters=2, random_state=42).fit(X_moons)
db = DBSCAN(eps=0.2, min_samples=5).fit(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=km.labels_, cmap="tab10", s=35, edgecolors="none")
axes[0].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1], c="red", marker="X", s=120, label="Centroids")
axes[0].set_title("K-Means on Interlocking Moons (Failure Mode)", fontweight="bold")
axes[0].legend()

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=db.labels_, cmap="tab10", s=35, edgecolors="none")
axes[1].set_title("DBSCAN on Interlocking Moons (Density Discovered)", fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Gaussian Mixture Models (GMM) & Soft Clustering
GMM models data as $K$ multivariate Gaussians with full covariance matrices, providing probabilistic responsibilities $\gamma_{ik} = P(z_i = k \mid \mathbf{x}_i)$ rather than hard assignments.

In [ ]:
# Generate anisotropic ellipsoidal clusters
np.random.seed(0)
X_aniso, _ = make_blobs(n_samples=300, centers=2, random_state=170)
transformation = [[0.6, -0.6], [-0.4, 0.8]]
X_aniso = np.dot(X_aniso, transformation)

gmm = GaussianMixture(n_components=2, covariance_type="full", random_state=42).fit(X_aniso)
probs = gmm.predict_proba(X_aniso)

plt.figure(figsize=(7, 4.5))
scatter = plt.scatter(X_aniso[:, 0], X_aniso[:, 1], c=probs[:, 1], cmap="viridis", s=40, edgecolors="k", linewidths=0.4)
plt.colorbar(scatter, label="Posterior Probability P(Cluster 1 | x)")
plt.title("GMM Soft Clustering Probabilities (Ellipsoidal Covariances)", fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Linear vs. Non-Linear Dimensionality Reduction: PCA vs. t-SNE
- **PCA**: Finds orthogonal linear projections preserving global maximum variance.
- **t-SNE**: Non-linear manifold learning minimizing KL divergence between high-dimensional Gaussian affinities and low-dimensional Student-t affinities, preserving local neighborhood structures.

In [ ]:
# High-dimensional synthetic manifold: 4 clusters in 20 dimensions
X_high, y_high = make_blobs(n_samples=400, n_features=20, centers=4, cluster_std=2.5, random_state=42)

pca_2d = PCA(n_components=2).fit_transform(X_high)
tsne_2d = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_high)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(pca_2d[:, 0], pca_2d[:, 1], c=y_high, cmap="tab10", s=35, edgecolors="none")
axes[0].set_title("PCA Linear Projection (Global Variance)", fontweight="bold")

axes[1].scatter(tsne_2d[:, 0], tsne_2d[:, 1], c=y_high, cmap="tab10", s=35, edgecolors="none")
axes[1].set_title("t-SNE Non-Linear Manifold (Local Neighborhoods)", fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Unsupervised Anomaly Detection with Isolation Forests
Isolation Forest isolates outliers by randomly selecting features and split values. Anomalies require fewer recursive splits to isolate (shorter tree paths).

In [ ]:
# Generate inliers and anomalies
np.random.seed(42)
X_inliers = 0.3 * np.random.randn(200, 2)
X_inliers = np.r_[X_inliers + 2, X_inliers - 2]
X_outliers = np.random.uniform(low=-4, high=4, size=(20, 2))
X_data = np.r_[X_inliers, X_outliers]

iso = IsolationForest(contamination=0.08, random_state=42).fit(X_data)
is_outlier = iso.predict(X_data) == -1

# Plot decision contour
xx, yy = np.meshgrid(np.linspace(-5, 5, 100), np.linspace(-5, 5, 100))
Z_iso = iso.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, Z_iso, levels=20, cmap="Blues_r", alpha=0.6)
plt.scatter(X_data[~is_outlier, 0], X_data[~is_outlier, 1], c="blue", s=30, label="Normal Inliers")
plt.scatter(X_data[is_outlier, 0], X_data[is_outlier, 1], c="red", s=60, marker="x", label="Detected Anomalies")
plt.title(f"Isolation Forest Anomaly Detection (Identified {np.sum(is_outlier)} outliers)", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()